# Week 08 Power BI Export Notebook — SkyOps Airline Delay Command Center

This notebook documents the Gold-to-Power BI hand-off for Week 8.

## 1. Purpose

The purpose of this notebook is to document the approved Gold tables used for the SkyOps Airline Delay Command Center Power BI dashboard and to provide simple reconciliation checks before/alongside the Power BI dashboard.

The Power BI dashboard uses the approved Gold tables only.

## 2. Gold Tables Used

The six Gold tables used in the Power BI dashboard are:

1. `Gold_Flight_Delay` — flight-level detail
2. `Gold_Airports` — airport reference
3. `Gold_Carriers` — carrier reference
4. `Gold_Routes` — route reference
5. `Gold_Carrier_Delay` — carrier + date daily aggregation
6. `Gold_Route_Daily` — route + date daily aggregation

## 3. Load Gold Data

The following cells load the Gold CSV files for reconciliation.



In [ ]:
from pathlib import Path
import pandas as pd

# Update this path only if your Gold CSV files are stored elsewhere.
gold_dir = Path('../data/gold')

files = {
    'Gold_Flight_Delay': '05_gold_aggregations.csv',
    'Gold_Airports': '05_gold_aggregations (1).csv',
    'Gold_Carriers': '05_gold_aggregations (2).csv',
    'Gold_Routes': '05_gold_aggregations (3).csv',
    'Gold_Carrier_Delay': '05_gold_aggregations (4).csv',
    'Gold_Route_Daily': '05_gold_aggregations (5).csv',
}

gold = {}
for table_name, file_name in files.items():
    path = gold_dir / file_name
    if path.exists():
        gold[table_name] = pd.read_csv(path)
        print(f'{table_name}: {gold[table_name].shape}')
    else:
        print(f'Not found: {path}')


## 4. Gold Source Register

The following table records the Gold source, declared grain, and dashboard purpose.


In [ ]:
source_register = pd.DataFrame([
    ['Gold_Flight_Delay', 'Flight-level detail', 'Flight operations and flight-level analysis'],
    ['Gold_Airports', 'Airport reference', 'Airport lookup and airport-related analysis'],
    ['Gold_Carriers', 'Carrier reference', 'Carrier lookup and carrier analysis'],
    ['Gold_Routes', 'Route reference', 'Route lookup and route analysis'],
    ['Gold_Carrier_Delay', 'Carrier + date daily', 'Carrier flight and cancellation KPIs'],
    ['Gold_Route_Daily', 'Route + date daily', 'Route performance KPIs and analysis'],
], columns=['Power BI Table', 'Declared Grain', 'Dashboard Purpose'])

source_register


## 5. Reconciliation Checks

These checks provide a simple source-level validation of the fields used by the Power BI measures.


In [ ]:
if 'Gold_Carrier_Delay' in gold:
    carrier_delay = gold['Gold_Carrier_Delay']
    required = ['total_flights', 'cancelled_flights']
    missing = [c for c in required if c not in carrier_delay.columns]
    if missing:
        print('Missing columns:', missing)
    else:
        total_flights = carrier_delay['total_flights'].sum()
        cancelled_flights = carrier_delay['cancelled_flights'].sum()
        cancellation_rate = cancelled_flights / total_flights if total_flights else 0
        print('Gold_Carrier_Delay reconciliation values:')
        print('Total Flights:', total_flights)
        print('Cancelled Flights:', cancelled_flights)
        print('Cancellation Rate:', cancellation_rate)
else:
    print('Gold_Carrier_Delay is not loaded. Check the gold_dir path above.')


## 6. Power BI Cancellation Rate Logic

The Power BI measure used on the Flight Operations Overview page is:

```DAX
Cancellation Rate =
DIVIDE(
    SUM(Gold_Carrier_Delay[cancelled_flights]),
    SUM(Gold_Carrier_Delay[total_flights])
)
```

The Python reconciliation above uses the same numerator and denominator from the Gold source.


In [ ]:
if 'Gold_Route_Daily' in gold:
    route_daily = gold['Gold_Route_Daily']
    required = ['total_flights', 'cancelled_flights']
    missing = [c for c in required if c not in route_daily.columns]
    if missing:
        print('Missing columns:', missing)
    else:
        route_total = route_daily['total_flights'].sum()
        route_cancelled = route_daily['cancelled_flights'].sum()
        route_rate = route_cancelled / route_total if route_total else 0
        print('Gold_Route_Daily reconciliation values:')
        print('Total Route Flights:', route_total)
        print('Cancelled Flights:', route_cancelled)
        print('Route Cancellation Rate:', route_rate)
else:
    print('Gold_Route_Daily is not loaded. Check the gold_dir path above.')


## 7. Power BI Route Cancellation Rate Logic

The Power BI measure used on the Route Performance page is:

```DAX
Route Cancellation Rate =
DIVIDE(
    SUM(Gold_Route_Daily[cancelled_flights]),
    SUM(Gold_Route_Daily[total_flights])
)
```

## 8. Week 8 Handoff

The approved Gold tables were handed off to Power BI for the Week 8 dashboard draft. The final dashboard is stored as `dashboard/powerbi_dashboard.pbix`. Dashboard evidence and the Week 8 log are stored in the corresponding repository folders.
